# Tree-sitter Parser Demonstration Notebook

This notebook demonstrates how the **`py-tree-sitter`** Python library parses Python code into a Concrete Syntax Tree (CST) using the official python bindings and python language grammar.

In [ ]:
# 1. Install dependencies if they are not already installed
# !pip install tree-sitter tree-sitter-python ipywidgets

In [ ]:
import os
import sys
from tree_sitter import Language, Parser
import tree_sitter_python as tspython

# 2. Initialize the parser
try:
    py_language = Language(tspython.language())
    parser = Parser()
    parser.language = py_language
except AttributeError:
    # Fallback for different py-tree-sitter versions
    parser = Parser()
    parser.set_language(Language(tspython.language()))
print("Tree-sitter Python parser loaded successfully!")

In [ ]:
# 3. Helper function to recursively print the tree nodes with visual lines
def print_tree(node, source_bytes, prefix="", is_last=True):
    node_text = source_bytes[node.start_byte:node.end_byte].decode('utf-8', errors='replace').strip()
    node_text_clean = node_text.replace('\n', ' ')
    if len(node_text_clean) > 40:
        node_text_clean = node_text_clean[:37] + "..."
    
    preview = f' "{node_text_clean}"' if node.child_count == 0 and node_text_clean else ""
    marker = "└── " if is_last else "├── "
    
    print(f"{prefix}{marker}{node.type} [{node.start_point[0]}:{node.start_point[1]} - {node.end_point[0]}:{node.end_point[1]}]{preview}")
    
    new_prefix = prefix + ("    " if is_last else "│   ")
    children = node.children
    for i, child in enumerate(children):
        child_is_last = (i == len(children) - 1)
        print_tree(child, source_bytes, new_prefix, child_is_last)

### Option A: Parse inline Python code snippet

In [ ]:
source_code = """
def calculate_sum(a, b):
    # Adds two numbers together
    result = a + b
    return result

print(calculate_sum(5, 10))
"""

source_bytes = source_code.encode('utf-8')
tree = parser.parse(source_bytes)

# Print starting from root node
root = tree.root_node
print(f"{root.type} [{root.start_point[0]}:{root.start_point[1]} - {root.end_point[0]}:{root.end_point[1]}]")
for i, child in enumerate(root.children):
    child_is_last = (i == len(root.children) - 1)
    print_tree(child, source_bytes, "", child_is_last)

### Option B: Upload a Python file to visualize its tree structure

Run the cell below to display an upload button. Select a `.py` file, and its tree structure will be parsed and displayed instantly.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

uploader = widgets.FileUpload(
    accept='.py',
    multiple=False,
    description="Upload .py File"
)

output_area = widgets.Output()

def on_upload_change(change):
    with output_area:
        clear_output()
        if not uploader.value:
            return
        
        # Handle difference in ipywidgets version format
        uploaded_file = list(uploader.value)[0] if isinstance(uploader.value, (list, tuple)) else uploader.value
        if isinstance(uploaded_file, str):
            file_info = uploader.value[uploaded_file]
            filename = uploaded_file
            content_bytes = file_info['content']
        else:
            filename = uploaded_file.get('name', 'uploaded_file.py')
            content_bytes = uploaded_file.get('content', b'')

        print(f"=== Successfully uploaded: {filename} ===")
        print(f"=== Raw content size: {len(content_bytes)} bytes ===\n")
        
        tree = parser.parse(content_bytes)
        root = tree.root_node
        print(f"{root.type} [{root.start_point[0]}:{root.start_point[1]} - {root.end_point[0]}:{root.end_point[1]}]")
        for i, child in enumerate(root.children):
            child_is_last = (i == len(root.children) - 1)
            print_tree(child, content_bytes, "", child_is_last)

uploader.observe(on_upload_change, names='value')

display(uploader)
display(output_area)

### Option C: Parse a Local File Path

Specify a local file path to parse and print its tree structure.

In [ ]:
local_file_path = "streamlit_app.py"  # Update this with any file path you want to view

if os.path.exists(local_file_path):
    with open(local_file_path, "rb") as f:
        file_bytes = f.read()
    
    print(f"=== Tree for Local File: {local_file_path} ===")
    tree = parser.parse(file_bytes)
    root = tree.root_node
    print(f"{root.type} [{root.start_point[0]}:{root.start_point[1]} - {root.end_point[0]}:{root.end_point[1]}]")
    for i, child in enumerate(root.children):
        child_is_last = (i == len(root.children) - 1)
        print_tree(child, file_bytes, "", child_is_last)
else:
    print(f"File '{local_file_path}' does not exist. Please check the path.")

### Option D: Run the Streamlit AST visualizer app

Alternatively, you can run the interactive Streamlit-based web dashboard. Open your terminal in this workspace directory and execute:

```bash
streamlit run tree_sitter_demo.py
```